## Chat Completions API

1. The simplest way to call an LLM
2. It's called Chat Completions because it's saying: "here is a conversation, please predict what should come next"
3. The Chat Completions API was invented by OpenAI, but it's so popular that everybody uses it!




In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


### Calling an http endpoint URL for respective models

In [2]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

{'model': 'gpt-5-nano',
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

In [3]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",    ##this is the endpoint URL for the OpenAI API
    headers=headers,
    json=payload
)

response.json()

{'id': 'chatcmpl-DrwAr2EmxpAIIaJQoGHMfxaDt2KTB',
 'object': 'chat.completion',
 'created': 1781747377,
 'model': 'gpt-5-nano-2025-08-07',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Did you know sea otters hold hands while they sleep to avoid drifting apart? They float in “rafts” and often wrap themselves in kelp to stay in place.',
    'refusal': None,
    'annotations': []},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 11,
  'completion_tokens': 683,
  'total_tokens': 694,
  'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 640,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint': None}

In [4]:
response.json()["choices"][0]["message"]["content"]

'Did you know sea otters hold hands while they sleep to avoid drifting apart? They float in “rafts” and often wrap themselves in kelp to stay in place.'

# What is the openai package?

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!


In [5]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content



'Fun fact: Octopuses have three hearts and blue blood—the two branchial hearts pump blood to the gills, while the third pumps it to the rest of the body. Want another fun fact?'

### Trying gemini

In [6]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")



No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini


In [7]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

BadRequestError: Error code: 400 - [{'error': {'code': 400, 'message': 'Please pass a valid API key', 'status': 'INVALID_ARGUMENT'}}]

### And Ollama also gives an OpenAI compatible endpoint

...and it's on my local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [8]:
requests.get("http://localhost:11434").content

b'Ollama is running'

### Download llama3.2:1b from meta


Don't use llama3.3 or llama4! They are too big for your computer..

In [18]:
!ollama pull llama3.2:1b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 74701a8c35f6:   0% ▕                  ▏ 2.1 MB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   0% ▕                  ▏ 3.7 MB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   1% ▕                  ▏ 8.6 MB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   1% ▕                  ▏  13 MB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   1% ▕                  ▏  15 MB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   2% ▕                  ▏  20 MB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   2% ▕                  ▏  25 MB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   2% ▕                  ▏  27 MB/1.3 GB                  pulling manifest 
pulling 

In [9]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [10]:
# Get a fun fact

response = ollama.chat.completions.create(model="llama3.2:1b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Here\'s a fun fact for you: There is a type of jellyfish that is immortal. The Turritopsis dohrnii, also known as the "immortal jellyfish," can transform its body into a younger state through a process called transdifferentiation, essentially making it immune to aging and death.'

###  Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

In [12]:


!ollama pull deepseek-r1:1.5b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling aabd4debf0c8: 100% ▕██████████████████▏ 1.1 GB                         
pulling c5ad996bda6e: 100% ▕██████████████████▏  556 B                         
pulling 6e4c38e1172f: 100% ▕██████████████████▏ 1.1 KB                         
pulling f4d24e9138dd: 100% ▕██████████████████▏  148 B                         
pulling a85fe2a2e58e: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


In [13]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'\n\nSure! Here\'s a fun fact for you:\n\nThe story of King Henry VIII, the first king of the United Kingdom, is often remembered from children’s movies because he was the longest-lived of all the Newton girls. But what makes this even more amusing than his famous lifespan? Well, he was so centimeters short that people used to ask him if he "had" a little foot.'

### Creating a website summarizer by calling an http endpoint of differnt models

In [14]:
from scraper import fetch_website_contents
from IPython.display import Markdown, display

In [15]:
system_prompt = """
You are a helpful assistant that analyzes the contents of a website,
and provides a serious short summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [16]:
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [17]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [18]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(model="llama3.2:1b",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [39]:
from openai import OpenAI
openai = OpenAI()

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(model="gpt-5-nano",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [19]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [20]:
display_summary("https://edwarddonner.com")

# Overview

The website features an article about Edward Donner, who is the co-founder and CTO of AI startup Nebula.io. The content appears to be a blog post where Donner shares his passion for Language Models (LLMs) and their potential. He also discusses his own expertise in AI engineering and provides resources and courses available on his website.

# Features

The article includes various sections, including:

*   **Introduction**: A brief introduction by Edward Donner himself.
*   **About Edward Donner**: Provides more information about the author's background and interests outside of AI.
*   **Posts**: There are several posts listed under different categories, offering insights into the topic of LLMs with personal anecdotes and experiences.
*   **Get in touch**: A contact form for individuals to reach out to Edward or follow his content on platforms such as LinkedIn, Twitter, and Facebook.

# Additional Information

There is a mention of courses available, including ones titled "AI Coder: Vibe Coder" and "Agentic Engineer". These topics may be related to the author's work in AI engineering. Additionally, social media profiles are provided for individuals to engage with Edward Donner and his expertise on various platforms.